# UniChart — Interpolation Methods & Table Reads

This notebook exercises **every `reg_order` interpolation option** supported by
`UnichartNotebook.table()` and demonstrates that a table read at an example
`x` value lands on the plotted regression curve.

For each method we:

1. Set the dataset's `reg_order` to the method under test.
2. Draw the scatter plot **with** that regression trendline.
3. Add a dashed **vertical `uc.line`** at the example `x` so the curve can be
   spot-checked by eye.
4. Display a **table** of the interpolated `y` read at that same `x`.

**Acceptance check:** the dashed vertical line should cross the trendline at
exactly the `y` value reported in the table.

## Setup

We use a positive, gently-curving dataset (`y ≈ 5·√x` plus noise) so that
every fit — including `log`, `exp`, and `power`, which require positive
values — is well-defined. A categorical `grade` column is included to show how
non-numeric columns are handled (nearest-x lookup rather than interpolation).

In [ ]:
# --- make repo-root importable (notebook lives in demo_notebooks/) ---
import sys, os
_repo_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import numpy as np
import pandas as pd
import unichart as uc_mod

np.random.seed(0)
x = np.linspace(1, 20, 25)
y = 5 * np.sqrt(x) + np.random.normal(0, 0.6, x.size)   # positive, smooth
mean_y = y.mean()
grade = np.where(y < mean_y - 3, 'low',
                 np.where(y < mean_y + 3, 'mid', 'high'))
df = pd.DataFrame({'x': x, 'y': y, 'grade': grade})

uc = uc_mod.UnichartNotebook()
uc.color_map = ['#E41A1C', '#377EB8', '#4DAF4A']
uc.marker_map = ['o', 's', 'D']

uc.load_df(df, title='demo')

uc.color(0, 9)
uc.marker(0, 9)


# Example x to read off each curve (in-range, not an existing data point).
X_EX = 7.3
df.head()

UniChart Notebook Environment Initialized.
Loaded Set 0: demo


,x,y,grade
0,1.000000,6.058431,low
1,1.791667,6.932752,low
2,2.583333,8.623618,low
3,3.375000,10.530122,low
4,4.166667,11.326742,low


In [ ]:
uc.reg_order?

Signature: uc.reg_order(uset_slice, order)
Docstring: Set a regression/trendline for the specified dataset(s).
File:      ~/Documents/GitHub/toms_utils/unichart.py
Type:      method

In [ ]:
uc.set_font_sizes?

Signature:
uc.set_font_sizes(
    suptitle=None,
    legend=None,
    axes_title=None,
    axes_tick=None,
    subplot_title=None,
    colorbar=None,
    hover=None,
    table_header=None,
    table_cell=None,
    all=None,
    reset=False,
)
Docstring:
Configure font sizes for plot and table elements. Settings persist across plots.

Parameters
----------
suptitle, legend, axes_title, axes_tick, subplot_title, colorbar, hover : float or str
    Font sizes for plot elements.
table_header : float or str
    Font size for table header row.
table_cell : float or str
    Font size for table cell content.
all : float or str
    Set all font sizes at once (overridden by individual parameters).
reset : bool
    Reset all font sizes to defaults.
File:      ~/Documents/GitHub/toms_utils/unichart.py
Type:      method

In [ ]:
def demo(reg, x_ex=X_EX):
    """Plot scatter + `reg` trendline, mark x_ex with a vertical line,
    and show the interpolated table read at x_ex."""
    uc.reg_order('all', reg)

    # Fresh vertical spot-check line at the example x.
    uc.line('all', 'clear')
    uc.line('x', x_ex)

    fig = uc.plot('x', 'y', suptitle=f"reg_order = {reg!r}")
    fig.show()

    # Numeric read used for the printed spot-check assertion.
    yval = float(uc.table(cols='y', x_col='x', x_in=x_ex, output='df')['y'].iloc[0])
    print(f"reg_order={reg!r}:  table read at x={x_ex}  ->  y = {yval:.4f}")
    print("Spot-check: the dashed vertical line should cross the trendline at this y.")

    # Styled display table (default output).
    uc.table(cols='y', x_col='x', x_in=x_ex, title=f"Interpolated read — {reg!r}")
    return yval

## `linear` — least-squares straight line

In [ ]:
demo('linear')

reg_order='linear':  table read at x=7.3  ->  y = 13.1074
Spot-check: the dashed vertical line should cross the trendline at this y.


Set,Dataset,x,y,INTERPOLATED
0,demo,7.3,13.107,True


13.107389435676449

## `poly2` — quadratic least-squares

In [ ]:
demo('poly2')

reg_order='poly2':  table read at x=7.3  ->  y = 13.5858
Spot-check: the dashed vertical line should cross the trendline at this y.


Set,Dataset,x,y,INTERPOLATED
0,demo,7.3,13.586,True


13.585812712203571

## `poly3` — cubic least-squares

In [ ]:
demo('poly3')

reg_order='poly3':  table read at x=7.3  ->  y = 13.9338
Spot-check: the dashed vertical line should cross the trendline at this y.


Set,Dataset,x,y,INTERPOLATED
0,demo,7.3,13.934,True


13.933808684673352

## `log` — logarithmic fit  (`y = a·ln(x) + b`)

In [ ]:
demo('log')

reg_order='log':  table read at x=7.3  ->  y = 14.9100
Spot-check: the dashed vertical line should cross the trendline at this y.


Set,Dataset,x,y,INTERPOLATED
0,demo,7.3,14.91,True


14.90998562870053

## `exp` — exponential fit  (`y = a·e^{bx}`)

In [ ]:
demo('exp')

reg_order='exp':  table read at x=7.3  ->  y = 12.3309
Spot-check: the dashed vertical line should cross the trendline at this y.


Set,Dataset,x,y,INTERPOLATED
0,demo,7.3,12.331,True


12.330897319729074

## `power` — power-law fit  (`y = a·x^b`)

In [ ]:
demo('power')

reg_order='power':  table read at x=7.3  ->  y = 13.9699
Spot-check: the dashed vertical line should cross the trendline at this y.


Set,Dataset,x,y,INTERPOLATED
0,demo,7.3,13.97,True


13.969947432973699

## `spline` — univariate spline (k=3)

In [ ]:
demo('spline')

reg_order='spline':  table read at x=7.3  ->  y = 13.9338
Spot-check: the dashed vertical line should cross the trendline at this y.


Set,Dataset,x,y,INTERPOLATED
0,demo,7.3,13.934,True


13.933808684673336

## `lowess` — locally-weighted smoothing

Requires `statsmodels`. If it is not installed the regression is skipped and
the table falls back to piecewise-linear interpolation.

In [ ]:
demo('lowess')

/home/tom/Documents/GitHub/toms_utils/unichart.py:739: UserWarning: LOWESS requires statsmodels. Install with `pip install statsmodels`.
  warnings.warn("LOWESS requires statsmodels. Install with `pip install statsmodels`.")


reg_order='lowess':  table read at x=7.3  ->  y = 13.4453
Spot-check: the dashed vertical line should cross the trendline at this y.


/home/tom/Documents/GitHub/toms_utils/unichart.py:739: UserWarning: LOWESS requires statsmodels. Install with `pip install statsmodels`.
  warnings.warn("LOWESS requires statsmodels. Install with `pip install statsmodels`.")
/home/tom/Documents/GitHub/toms_utils/unichart.py:739: UserWarning: LOWESS requires statsmodels. Install with `pip install statsmodels`.
  warnings.warn("LOWESS requires statsmodels. Install with `pip install statsmodels`.")


Set,Dataset,x,y,INTERPOLATED
0,demo,7.3,13.445,True


13.445265076554602

## `ma` — centered moving average

In [ ]:
demo('ma')

reg_order='ma':  table read at x=7.3  ->  y = 13.5309
Spot-check: the dashed vertical line should cross the trendline at this y.


Set,Dataset,x,y,INTERPOLATED
0,demo,7.3,13.531,True


13.530857530574002

## Tuple specs — explicit `(kind, param)`

`reg_order` (and the `table(kind=...)` override) also accept `(kind, param)`
tuples for full control over the fit parameter.

In [ ]:
demo(('poly', 4))   # quartic

reg_order=('poly', 4):  table read at x=7.3  ->  y = 13.9435
Spot-check: the dashed vertical line should cross the trendline at this y.


Set,Dataset,x,y,INTERPOLATED
0,demo,7.3,13.943,True


13.94347268461748

In [ ]:
demo(('spline', 5))   # quintic spline

reg_order=('spline', 5):  table read at x=7.3  ->  y = 13.6587
Spot-check: the dashed vertical line should cross the trendline at this y.


Set,Dataset,x,y,INTERPOLATED
0,demo,7.3,13.659,True


13.65873193029836

In [ ]:
demo(('lowess', 0.5))   # wider smoothing window

/home/tom/Documents/GitHub/toms_utils/unichart.py:739: UserWarning: LOWESS requires statsmodels. Install with `pip install statsmodels`.
  warnings.warn("LOWESS requires statsmodels. Install with `pip install statsmodels`.")


/home/tom/Documents/GitHub/toms_utils/unichart.py:739: UserWarning: LOWESS requires statsmodels. Install with `pip install statsmodels`.
  warnings.warn("LOWESS requires statsmodels. Install with `pip install statsmodels`.")
/home/tom/Documents/GitHub/toms_utils/unichart.py:739: UserWarning: LOWESS requires statsmodels. Install with `pip install statsmodels`.
  warnings.warn("LOWESS requires statsmodels. Install with `pip install statsmodels`.")


reg_order=('lowess', 0.5):  table read at x=7.3  ->  y = 13.4453
Spot-check: the dashed vertical line should cross the trendline at this y.


Set,Dataset,x,y,INTERPOLATED
0,demo,7.3,13.445,True


13.445265076554602

## Per-call `kind` override

`table(kind=...)` overrides the dataset's `reg_order` for that read only,
accepting the same specs. Here the dataset stays on `poly2` but we read the
value off a `power` fit instead.

In [ ]:
uc.reg_order('all', 'poly2')
read_poly2 = float(uc.table(cols='y', x_col='x', x_in=X_EX, output='df')['y'].iloc[0])
read_power = float(uc.table(cols='y', x_col='x', x_in=X_EX, kind='power', output='df')['y'].iloc[0])
print(f"dataset reg_order='poly2' read : {read_poly2:.4f}")
print(f"kind='power' override read     : {read_power:.4f}")

dataset reg_order='poly2' read : 13.5858
kind='power' override read     : 13.9699


## No regression — piecewise-linear fallback

With `reg_order` unset, the table interpolates piecewise-linearly **through the
raw points** (the original default behaviour), so the read lands exactly on the
straight segment connecting the two neighbouring data points.

In [ ]:
uc.reg_order('all', None)
uc.line('all', 'clear')
uc.line('x', X_EX)
uc.plot('x', 'y', suptitle='reg_order = None  (piecewise-linear table read)').show()

# Compare the table read to a manual numpy piecewise-linear interpolation.
manual = float(np.interp(X_EX, df['x'], df['y']))
read = float(uc.table(cols='y', x_col='x', x_in=X_EX, output='df')['y'].iloc[0])
print(f"table read           : {read:.4f}")
print(f"np.interp (expected) : {manual:.4f}")
assert np.isclose(read, manual), 'piecewise fallback mismatch!'
print('OK — piecewise fallback matches np.interp')
uc.table(cols='y', x_col='x', x_in=X_EX, title='Piecewise-linear read')

table read           : 13.4453
np.interp (expected) : 13.4453
OK — piecewise fallback matches np.interp


Set,Dataset,x,y,INTERPOLATED
0,demo,7.3,13.445,True


## Multiple x reads + non-numeric column

The table accepts a list of `x_in` values and also carries non-numeric columns
(here `grade`) by taking the value from the row whose `x` is nearest each
requested point. The `INTERPOLATED` column flags reads that fall between data
points.

In [ ]:
uc.reg_order('all', 'power')
uc.table(cols=['y', 'grade'], x_col='x', x_in=[3.0, 7.3, 12.5, 18.0],
         title='Multiple reads (power fit) with categorical column')

Set,Dataset,x,y,grade,INTERPOLATED
0,demo,3,9.3247,low,True
0,demo,7.3,13.97,mid,True
0,demo,12.5,17.839,mid,True
0,demo,18,21.055,high,True


## Output modes — `df` and `md`

`output='df'` returns the assembled DataFrame; `output='md'` returns a Markdown
string (requires the `tabulate` package).

In [ ]:
out_df = uc.table(cols=['y', 'grade'], x_col='x', x_in=[3.0, 7.3, 12.5],
                  output='df')
print(type(out_df))
out_df

<class 'pandas.DataFrame'>


,Set,Dataset,x,y,grade,INTERPOLATED
0,0,demo,3.0,9.324701,low,True
1,0,demo,7.3,13.969947,mid,True
2,0,demo,12.5,17.839259,mid,True


In [ ]:
md = uc.table(cols=['y', 'grade'], x_col='x', x_in=[3.0, 7.3, 12.5],
              output='md')
print(md)

Markdown output requires the 'tabulate' package (pip install tabulate).
None
